<a href="https://colab.research.google.com/github/gflamiano/SJU-CSC-672-ML/blob/main/HW4/HW4_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 4 Analysis

#### 1. How do the feature types in the UCI Heart Disease dataset (combining continuous physiological telemetry like `chol` and `thalach` with categorical/ordinal indicators like `cp` and `thal`) differ from previous datasets, and why is proper scaling critical for Support Vector Machines compared to Decision Trees?

The dataset combines continuous measurements such as `age`, `chol`, and `thalach` with categorical indicators such as `cp` and `thal`, yet all of them are stored as numbers. As a result, pandas cannot distinguish the feature types, so I assigned them manually, scaling the continuous features and one-hot encoding the categorical ones. The continuous features also differ greatly in scale, with `chol` reaching about 564 while `oldpeak` reaches only about 6. SVMs rely on distances between data points, so a large-valued feature like `chol` would dominate the smaller ones unless all features are scaled. Decision Trees split on one feature at a time using thresholds, so the magnitude of the values does not affect their splits. Scaling is therefore essential for the SVC but unnecessary for the Decision Tree.

#### 2. Explain how hidden missing values (such as `?` symbols) impact raw medical datasets, and justify your choice of imputation strategies for skewed numerical vitals versus categorical diagnostic fields.

Raw medical datasets often encode missing values as placeholders such as `?`, which converts numeric columns into text and can cause errors or silently distort calculations. In this dataset, only `ca` (4 missing) and `thal` (2 missing) contained gaps, and the placeholder check found no additional hidden values. I used median imputation for numerical features because variables such as `oldpeak` and `chol` are skewed and contain outliers that would pull a mean upward. I used most-frequent imputation for categorical features such as `ca` and `thal`, because a mean could produce a category that does not exist, such as `thal = 5.3`. One limitation is that I imputed before splitting the data into training and test sets, which introduces a minor form of data leakage. Because only six values were imputed, the effect is likely negligible.

#### 3. Contrast how a Support Vector Classifier (using an RBF kernel) constructs decision boundaries versus how a Decision Tree recursively partitions feature space. What are the interpretability tradeoffs between the two models?

The SVC constructs a smooth, curved decision boundary that uses all features together to separate patients with and without heart disease. The Decision Tree instead partitions the data through a sequence of simple questions, such as whether the `thal` result is normal, splitting patients into progressively smaller groups. Because the tree splits on only one feature at a time, it requires many small splits to approximate a curved boundary. The tree is considerably more interpretable, as each decision path can be followed and explained in terms resembling clinical reasoning. The SVC, by contrast, functions largely as a black box whose predictions are difficult to explain. Overall, the SVC tends to be smoother and more stable, while the tree is easier to interpret but more sensitive to changes in the training data.

#### 4. Discuss how tuning hyperparameters such as regularization parameter `C` and kernel coefficient `gamma` in an SVC impacts the trade-off between bias, variance, and model generalization on unseen medical test data.

The parameter `C` controls how strictly the model tries to classify every training point correctly: a small `C` is more lenient and risks underfitting, while a large `C` is stricter and risks overfitting. The parameter `gamma` controls how far the influence of each data point extends: a small `gamma` produces a smooth boundary that may underfit, while a large `gamma` produces a highly flexible boundary that may overfit. My grid search selected `C = 10` and `gamma = 0.001`, with a cross-validation ROC-AUC of about 0.895, indicating that a relatively smooth boundary performed best. Because `0.001` was the smallest value tested, an even smaller `gamma` might perform as well or better. Tuning was performed on the training data only, which kept the test set untouched for an unbiased evaluation. On the test set, both the untuned and tuned SVC achieved a ROC-AUC of 0.964, meaning tuning did not improve test performance, and both clearly outperformed the Decision Tree at 0.744. The Decision Tree's weaker result is consistent with the high variance of an unpruned tree, which can overfit the training data. The test score being higher than the cross-validation score is likely due to the small dataset, since the test set contains only 61 patients and differences between models of a few points may not be meaningful.



#### 5. Inspecting your **Decision Tree feature importances** or model coefficients, which clinical indicators (e.g., maximum heart rate, chest pain type, ST depression) exhibit the strongest predictive power for heart disease, and what preventative screening insights do these results provide for cardiologists?

The most important feature in the decision tree was `thal = 3.0` (0.293), which corresponds to a normal thallium stress test result. It was followed by `cp = 4.0` (asymptomatic chest pain, 0.119), `chol` (0.110), `oldpeak` (0.108), `ca = 0` (0.098), and `age` (0.096). The prominence of the `thal` result, `ca`, and `oldpeak` is consistent with their direct relationship to blood flow and cardiac stress. The high ranking of asymptomatic chest pain suggests that patients without chest pain symptoms may still have heart disease. The high ranking of `chol` is somewhat surprising, but decision trees tend to favor continuous features, so this does not necessarily indicate that cholesterol is a stronger predictor. For cardiologists, these results suggest that stress testing and vessel imaging may be more informative than symptoms alone, particularly for patients who have risk factors but no symptoms. These results come from a single tree trained on 303 patients, so the rankings may change with a different data split, and they reflect what the model used rather than what causes heart disease.